# Prédiction du Churn Orbitel & Optimisation de Décision Business

Ce notebook présente la modélisation prédictive du churn (départ des clients) pour l'entreprise **Orbitel**.

L'originalité de ce projet réside dans le fait que nous n'optimisons pas uniquement des métriques statistiques classiques (comme l'accuracy ou le F1-score), mais nous cherchons à **minimiser le coût financier global** pour l'entreprise.

## Problématique Métier
Chaque type d'erreur de prédiction possède un coût financier distinct :
- **Faux Négatif (FN)** : Le modèle prédit qu'un client restera fidèle alors qu'il va partir $\rightarrow$ **Coût = 200 €** (perte totale du client sans action de rétention).
- **Faux Positif (FP)** : Le modèle prédit qu'un client va partir alors qu'il est fidèle $\rightarrow$ **Coût = 50 €** (dépense marketing inutile pour une offre promotionnelle).

La fonction de coût métier s'écrit :
$$\text{Coût Métier} = 200 \times \text{FN} + 50 \times \text{FP}$$

Notre objectif est de minimiser cette fonction.

## 1. Importation des librairies et configuration

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Importations depuis notre module local
from src.model import (
    load_data,
    add_frustration_feature,
    train_baseline_model,
    train_improved_model,
    evaluate_model,
    optimize_threshold
)
from run_analysis import prepare_features

sns.set_theme(style="whitegrid")
print("Librairies chargées avec succès.")

## 2. Chargement des données ou génération de données de simulation
Si les fichiers `churn_train.csv`, `churn_test.csv` et `churn_new.csv` ne sont pas encore présents dans le dossier, nous générons des données simulées pour tester le code.

In [ ]:
train_path = 'churn_train.csv'
test_path = 'churn_test.csv'
new_path = 'churn_new.csv'

if not (os.path.exists(train_path) and os.path.exists(test_path)):
    print("Données réelles non détectées. Génération de données synthétiques pour démonstration...")
    from test_pipeline import generate_synthetic_data
    
    df_train = generate_synthetic_data(n_samples=1000, random_seed=42)
    df_test = generate_synthetic_data(n_samples=300, random_seed=100)
    df_new = generate_synthetic_data(n_samples=150, random_seed=999).drop(columns=['Churn_Y'])
    
    df_train.to_csv(train_path, index=False, sep=';')
    df_test.to_csv(test_path, index=False, sep=';')
    df_new.to_csv(new_path, index=False, sep=';')
else:
    print("Données réelles détectées. Chargement des fichiers en cours...")
    df_train = load_data(train_path)
    df_test = load_data(test_path)
    df_new = load_data(new_path) if os.path.exists(new_path) else None

print(f"Jeu d'entraînement : {df_train.shape}")
print(f"Jeu de validation   : {df_test.shape}")

## 3. Partie 1 : Modèle Initial (Régression Logistique de Base)
Nous commençons par préparer les données brutes sans transformation et entraînons un modèle initial avec les paramètres par défaut.

In [ ]:
X_train, y_train, X_test, y_test, X_new = prepare_features(df_train, df_test, df_new)

print("Entraînement du modèle de base...")
baseline_model = train_baseline_model(X_train, y_train)

# Évaluation au seuil par défaut de 0.5
baseline_metrics = evaluate_model(baseline_model, X_test, y_test, threshold=0.5)

print(f"Coût Métier Initial (Seuil 0.5) : {baseline_metrics['total_cost']:.2f} €")
print(f"Structure des erreurs : FN = {baseline_metrics['FN']} ({baseline_metrics['fn_cost']:.0f} €), FP = {baseline_metrics['FP']} ({baseline_metrics['fp_cost']:.0f} €)")

## 4. Partie 2 : Amélioration du Modèle (Standardisation & Gestion du Déséquilibre)
Nous appliquons les améliorations techniques :
- Standardisation des données (`StandardScaler`) pour accélérer la convergence (et faire disparaître le warning de limite d'itérations).
- Équilibrage des classes (`class_weight='balanced'`) pour pénaliser les erreurs sur la classe minoritaire (le churn).

In [ ]:
print("Entraînement du modèle amélioré...")
improved_model = train_improved_model(X_train, y_train, class_weight='balanced')
improved_metrics = evaluate_model(improved_model, X_test, y_test, threshold=0.5)

print(f"Coût Métier Modèle Amélioré (Seuil 0.5) : {improved_metrics['total_cost']:.2f} €")
print(f"Structure des erreurs : FN = {improved_metrics['FN']} ({improved_metrics['fn_cost']:.0f} €), FP = {improved_metrics['FP']} ({improved_metrics['fp_cost']:.0f} €)")
print(f"Gain financier : {baseline_metrics['total_cost'] - improved_metrics['total_cost']:.2f} €")

## 5. Partie 3 : Variable 'Frustration' et Optimisation du Seuil
Nous intégrons la variable métier `'frustration'` et cherchons le seuil optimal entre 0.1 et 0.9 pour minimiser le coût financier final.

In [ ]:
# Création de la variable frustration
df_train_frust = add_frustration_feature(df_train)
df_test_frust = add_frustration_feature(df_test)
df_new_frust = add_frustration_feature(df_new) if df_new is not None else None

X_train_frust, y_train, X_test_frust, y_test, X_new_frust = prepare_features(df_train_frust, df_test_frust, df_new_frust)

# Entraînement du modèle final
final_model = train_improved_model(X_train_frust, y_train, class_weight='balanced')

# Recherche du seuil optimal
best_threshold, best_metrics, df_thresholds = optimize_threshold(final_model, X_test_frust, y_test)
df_thresholds

### Visualisation des résultats
Tracé du coût métier total en fonction du seuil de classification.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df_thresholds['threshold'], df_thresholds['total_cost'], marker='o', linewidth=2.5, color='#e74c3c', label='Coût métier total')
plt.plot(df_thresholds['threshold'], df_thresholds['fn_cost'], '--', color='#2ecc71', label='Coût Faux Négatifs (Clients perdus)')
plt.plot(df_thresholds['threshold'], df_thresholds['fp_cost'], '--', color='#3498db', label='Coût Faux Positifs (Campagnes inutiles)')

# Indiquer le seuil optimal
plt.axvline(x=best_threshold, color='#8e44ad', linestyle=':', linewidth=2, label=f'Seuil Optimal ({best_threshold:.1f})')
plt.scatter(best_threshold, best_metrics['total_cost'], color='#8e44ad', s=150, zorder=5)

plt.title("Évolution du Coût Métier Global en fonction du Seuil de Décision", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Seuil de Décision (Classification Threshold)", fontsize=12)
plt.ylabel("Coût Financier pour l'entreprise (€)", fontsize=12)
plt.xticks(np.arange(0.1, 1.0, 0.1))
plt.legend(fontsize=11, loc='upper center')
plt.tight_layout()
plt.show()

## 6. Génération des prédictions finales
Nous appliquons notre meilleur modèle avec le seuil optimal sur les nouveaux clients.

In [ ]:
if X_new_frust is not None:
    probs_new = final_model.predict_proba(X_new_frust)[:, 1]
    preds_new = (probs_new >= best_threshold).astype(int)
    
    df_predictions = pd.DataFrame({
        'Probabilite_Churn': probs_new,
        'Pred_Churn_Y': preds_new
    })
    
    # Insérer l'ID s'il existe
    id_cols = [col for col in df_new.columns if any(kw in col.lower() for kw in ['id', 'client', 'customer'])]
    if id_cols:
        df_predictions.insert(0, id_cols[0], df_new[id_cols[0]])
        
    output_file = 'predictions_finales.csv'
    df_predictions.to_csv(output_file, index=False, sep=';')
    print(f"Prédictions enregistrées dans '{output_file}'")
    print(f"Nombre de churneurs détectés : {sum(preds_new)} sur {len(preds_new)} clients.")
else:
    print("Fichier churn_new.csv non disponible.")